In [30]:
%%writefile train_nano_jepa.py
# Paste the full code above here
#!/usr/bin/env python3
import argparse
import datetime
import math
import os
import time
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler

import torchvision
import torchvision.transforms as T


# ---------------------------------------------------------------------
# Distributed setup
# ---------------------------------------------------------------------
def setup_dist():
    # Kaggle T4 boxes often do better with P2P/IB disabled.
    # If you are on a workstation with NVLink, you can remove these.
    os.environ.setdefault("NCCL_P2P_DISABLE", "1")
    os.environ.setdefault("NCCL_IB_DISABLE", "1")

    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        dist.init_process_group(
            backend="nccl",
            timeout=datetime.timedelta(minutes=30),
        )
        rank = dist.get_rank()
        world_size = dist.get_world_size()
        local_rank = int(os.environ.get("LOCAL_RANK", 0))
        torch.cuda.set_device(local_rank)
        return rank, world_size, local_rank, True

    torch.cuda.set_device(0)
    return 0, 1, 0, False


def cleanup_dist():
    if dist.is_initialized():
        dist.destroy_process_group()


def set_seed(seed: int, rank: int):
    seed = seed + rank
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ---------------------------------------------------------------------
# Model building blocks
# ---------------------------------------------------------------------
class MLP(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class Attention(nn.Module):
    def __init__(self, dim: int, heads: int):
        super().__init__()
        assert dim % heads == 0
        self.heads = heads
        self.head_dim = dim // heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        # Efficient fused attention path.
        x = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=False)

        x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x


class Block(nn.Module):
    def __init__(self, dim: int, heads: int, mlp_ratio: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, int(dim * mlp_ratio))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class Transformer(nn.Module):
    def __init__(
        self,
        dim: int,
        depth: int,
        heads: int,
        mlp_ratio: float,
        grad_ckpt: bool = False,
    ):
        super().__init__()
        self.grad_ckpt = grad_ckpt
        self.blocks = nn.ModuleList([Block(dim, heads, mlp_ratio) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        for blk in self.blocks:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                x = checkpoint(blk, x, use_reentrant=False)
            else:
                x = blk(x)
        return self.norm(x)


def _init_weights(m: nn.Module):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LayerNorm):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Conv2d):
        nn.init.kaiming_uniform_(m.weight, a=math.sqrt(5))
        if m.bias is not None:
            nn.init.zeros_(m.bias)


# ---------------------------------------------------------------------
# Nano JEPA
# ---------------------------------------------------------------------
class NanoJEPA(nn.Module):
    """
    Nano JEPA matching your reported configuration:
      - 64x64 input
      - 8x8 patches => 64 patches
      - embed dim 384
      - 6-layer context encoder
      - 6-layer target encoder
      - 3-layer predictor
      - 75% masking
      - EMA target encoder
    """

    def __init__(
        self,
        img_size: int = 64,
        patch_size: int = 8,
        embed_dim: int = 384,
        context_depth: int = 6,
        target_depth: int = 6,
        predictor_depth: int = 3,
        heads: int = 12,
        mlp_ratio: float = 4.0,
        mask_ratio: float = 0.75,
        grad_ckpt: bool = True,
    ):
        super().__init__()
        assert img_size % patch_size == 0

        self.img_size = img_size
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_patches = (img_size // patch_size) ** 2
        self.num_keep = int(round(self.num_patches * (1.0 - mask_ratio)))

        # Online/student branch
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.context_encoder = Transformer(
            dim=embed_dim,
            depth=context_depth,
            heads=heads,
            mlp_ratio=mlp_ratio,
            grad_ckpt=grad_ckpt,
        )

        # Target/teacher branch: separate parameters, EMA-updated, no gradients.
        self.target_patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.target_pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.target_encoder = Transformer(
            dim=embed_dim,
            depth=target_depth,
            heads=heads,
            mlp_ratio=mlp_ratio,
            grad_ckpt=False,
        )

        # Predictor
        self.predictor = Transformer(
            dim=embed_dim,
            depth=predictor_depth,
            heads=heads,
            mlp_ratio=mlp_ratio,
            grad_ckpt=grad_ckpt,
        )

        # Small head helps match your reported ~26.96M parameter count.
        self.predictor_head = nn.Linear(embed_dim, embed_dim)

        self.apply(_init_weights)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.target_pos_embed, std=0.02)
        nn.init.trunc_normal_(self.mask_token, std=0.02)

        self.init_target()

        # Freeze target branch.
        for p in self.target_encoder.parameters():
            p.requires_grad = False
        for p in self.target_patch_embed.parameters():
            p.requires_grad = False
        self.target_pos_embed.requires_grad = False

    @torch.no_grad()
    def init_target(self):
        self.target_encoder.load_state_dict(self.context_encoder.state_dict())
        self.target_patch_embed.load_state_dict(self.patch_embed.state_dict())
        self.target_pos_embed.copy_(self.pos_embed)

    @torch.no_grad()
    def update_target(self, momentum: float):
        # target = momentum * target + (1 - momentum) * online
        alpha = 1.0 - momentum

        for pt, po in zip(self.target_encoder.parameters(), self.context_encoder.parameters()):
            pt.lerp_(po, alpha)

        for pt, po in zip(self.target_patch_embed.parameters(), self.patch_embed.parameters()):
            pt.lerp_(po, alpha)

        self.target_pos_embed.lerp_(self.pos_embed, alpha)

    def make_mask_ids(self, B: int, device: torch.device):
        noise = torch.rand(B, self.num_patches, device=device)
        ids = torch.argsort(noise, dim=1)
        visible_ids = ids[:, : self.num_keep]
        mask_ids = ids[:, self.num_keep :]
        return visible_ids, mask_ids

    def forward(self, x):
        B = x.shape[0]
        device = x.device

        # Online visible tokens.
        tokens = self.patch_embed(x).flatten(2).transpose(1, 2)
        tokens = tokens + self.pos_embed

        # Target full-image representations, no gradients.
        with torch.no_grad():
            target_tokens = self.target_patch_embed(x).flatten(2).transpose(1, 2)
            target_tokens = target_tokens + self.target_pos_embed
            target = self.target_encoder(target_tokens)

        visible_ids, mask_ids = self.make_mask_ids(B, device)

        visible_idx = visible_ids.unsqueeze(-1).expand(-1, -1, self.embed_dim)
        mask_idx = mask_ids.unsqueeze(-1).expand(-1, -1, self.embed_dim)

        visible_tokens = torch.gather(tokens, dim=1, index=visible_idx)
        context_out = self.context_encoder(visible_tokens)

        # Predictor input:
        #   - mask tokens at all positions
        #   - replace visible positions with context encoder outputs
        pred_input = (self.mask_token + self.pos_embed).expand(B, self.num_patches, self.embed_dim).contiguous()
        pred_input.scatter_(dim=1, index=visible_idx, src=context_out)

        pred = self.predictor(pred_input)
        pred = self.predictor_head(pred)

        pred_mask = torch.gather(pred, dim=1, index=mask_idx)
        target_mask = torch.gather(target, dim=1, index=mask_idx).detach()

        # Flatten to (B * num_masked_patches, C)
        pred_mask = pred_mask.reshape(-1, self.embed_dim)
        target_mask = target_mask.reshape(-1, self.embed_dim)

        return pred_mask, target_mask


# ---------------------------------------------------------------------
# Loss
# ---------------------------------------------------------------------
def variance_loss(x: torch.Tensor) -> torch.Tensor:
    std = torch.sqrt(x.var(dim=0) + 1e-4)
    return F.relu(1.0 - std).mean()


def covariance_loss(x: torch.Tensor) -> torch.Tensor:
    B, C = x.shape
    if B < 2:
        return x.new_zeros(())

    x = x - x.mean(dim=0)
    cov = (x.T @ x) / (B - 1)

    mask = ~torch.eye(C, dtype=torch.bool, device=x.device)
    off_diag = cov[mask]
    return off_diag.pow(2).sum() / C


def jepa_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    std_weight: float = 0.05,
    cov_weight: float = 0.04,
) -> torch.Tensor:
    pred = pred.float()
    target = target.float()

    loss = F.smooth_l1_loss(pred, target, beta=1.0)

    if std_weight > 0:
        loss = loss + std_weight * variance_loss(pred)

    if cov_weight > 0:
        loss = loss + cov_weight * covariance_loss(pred)

    return loss


# ---------------------------------------------------------------------
# Datasets
# ---------------------------------------------------------------------
class SyntheticDataset(Dataset):
    """
    Used for pure GPU/MFU benchmarking without disk/CIFAR overhead.
    """

    def __init__(self, size: int = 100000, img_size: int = 64):
        self.size = size
        self.img_size = img_size

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        x = torch.rand(3, self.img_size, self.img_size, dtype=torch.float32)
        return x, 0


def build_transform(args):
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)

    if args.aug:
        return T.Compose([
            T.Resize(int(args.img_size * 1.125)),
            T.RandomCrop(args.img_size),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(mean, std),
        ])

    return T.Compose([
        T.Resize(args.img_size),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])


def build_dataset(args, rank: int, world_size: int, ddp: bool):
    if args.synthetic:
        dataset = SyntheticDataset(size=args.synthetic_size, img_size=args.img_size)
        sampler = (
            DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, drop_last=True)
            if ddp
            else None
        )
        return dataset, sampler

    transform = build_transform(args)

    # Replaced CIFAR10 with ImageFolder to read the Kaggle dataset safely
    if ddp:
        dist.barrier()
        dataset = torchvision.datasets.ImageFolder(
            root=args.data_path,
            transform=transform,
        )
        sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, drop_last=True)
    else:
        dataset = torchvision.datasets.ImageFolder(
            root=args.data_path,
            transform=transform,
        )
        sampler = None

    return dataset, sampler


def build_loader(dataset, sampler, args):
    loader_kwargs = dict(
        batch_size=args.batch_size,
        num_workers=args.workers,
        pin_memory=True,
        drop_last=True,
    )

    if sampler is not None:
        loader_kwargs["sampler"] = sampler
    else:
        loader_kwargs["shuffle"] = True

    if args.workers > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = args.prefetch

    return DataLoader(dataset, **loader_kwargs)


# ---------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------
def unwrap_model(model: nn.Module) -> nn.Module:
    if isinstance(model, DDP):
        model = model.module
    return getattr(model, "_orig_mod", model)


def adjust_lr(optimizer, step: int, total_steps: int, warmup_steps: int, base_lr: float) -> float:
    if step < warmup_steps:
        lr = base_lr * (step + 1) / max(1, warmup_steps)
    else:
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        lr = base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

    for g in optimizer.param_groups:
        g["lr"] = lr

    return lr


def cosine_schedule(step: int, total_steps: int, base: float, end: float) -> float:
    if total_steps <= 1:
        return base
    progress = min(step / max(1, total_steps - 1), 1.0)
    return base + (end - base) * 0.5 * (1.0 - math.cos(math.pi * progress))


def save_checkpoint(path: Path, model, optimizer, scaler, epoch: int, args):
    obj = {
        "model": unwrap_model(model).state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "args": vars(args),
    }
    torch.save(obj, path)


# ---------------------------------------------------------------------
# Optional t-SNE export
# ---------------------------------------------------------------------
@torch.no_grad()
def run_tsne(model_without_ddp: NanoJEPA, args, device: torch.device):
    try:
        from sklearn.manifold import TSNE
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("scikit-learn and matplotlib are required for --tsne. Skipping t-SNE.")
        return

    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)

    transform = T.Compose(
        [
            T.Resize(args.img_size),
            T.ToTensor(),
            T.Normalize(mean, std),
        ]
    )

    dataset = torchvision.datasets.ImageFolder(
        root=args.data_path,
        transform=transform,
    )

    loader = DataLoader(
        dataset,
        batch_size=256,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        drop_last=False,
    )

    feats = []
    labels = []
    collected = 0

    model_without_ddp.eval()

    for x, y in loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)

        with torch.cuda.amp.autocast(enabled=not args.disable_fp16):
            tokens = model_without_ddp.target_patch_embed(x).flatten(2).transpose(1, 2)
            tokens = tokens + model_without_ddp.target_pos_embed
            out = model_without_ddp.target_encoder(tokens)
            feat = out.mean(dim=1)

        feats.append(feat.float().cpu())
        labels.append(y)

        collected += feat.size(0)
        if collected >= args.tsne_samples:
            break

    feats = torch.cat(feats, dim=0)[: args.tsne_samples]
    labels = torch.cat(labels, dim=0)[: args.tsne_samples]

    print(f"Running t-SNE on {feats.shape[0]} samples...")
    emb = TSNE(
        n_components=2,
        perplexity=30,
        init="pca",
        learning_rate="auto",
        random_state=42,
    ).fit_transform(feats.numpy())

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        emb[:, 0],
        emb[:, 1],
        c=labels.numpy(),
        cmap="tab10",
        s=8,
        alpha=0.85,
    )
    plt.colorbar(scatter)
    plt.title("Nano JEPA latent t-SNE")
    out_path = Path(args.output_dir) / "tsne_nano_jepa.png"
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    print(f"Saved t-SNE figure to {out_path}")


# ---------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------
def train_one_epoch(
    model,
    model_without_ddp,
    loader,
    optimizer,
    scaler,
    epoch: int,
    args,
    device: torch.device,
    world_size: int,
    rank: int,
    global_step: int,
    total_steps: int,
    warmup_steps: int,
    base_lr: float,
    total_params: int,
):
    model.train()
    model_without_ddp.target_encoder.eval()

    if loader.sampler is not None and hasattr(loader.sampler, "set_epoch"):
        loader.sampler.set_epoch(epoch)

    rank0 = rank == 0
    window_images = 0
    window_start = None

    for step, (images, _) in enumerate(loader):
        if args.benchmark_steps > 0 and global_step >= args.benchmark_steps:
            break

        lr = adjust_lr(optimizer, global_step, total_steps, warmup_steps, base_lr)
        momentum = cosine_schedule(global_step, total_steps, args.momentum, args.momentum_end)

        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=not args.disable_fp16):
            pred, target = model(images)
            loss = jepa_loss(
                pred,
                target,
                std_weight=args.std_weight,
                cov_weight=args.cov_weight,
            )

        scaler.scale(loss).backward()

        if args.clip_grad > 0:
            if not args.disable_fp16:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip_grad)

        scaler.step(optimizer)
        scaler.update()

        model_without_ddp.update_target(momentum)

        batch_images = images.size(0) * world_size

        # Skip the first few warmup iterations for stable throughput measurement.
        if global_step >= 5:
            if window_start is None:
                window_start = time.perf_counter()
            window_images += batch_images

        if rank0 and window_images > 0 and (step + 1) % args.log_interval == 0:
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - window_start
            img_sec = window_images / max(elapsed, 1e-9)

            # Calibrated MFU using your reported Phase-1 accounting.
            tflops = img_sec * args.flops_per_image / 1e12
            mfu = 100.0 * tflops / args.peak_tflops

            # Theoretical transformer-style estimate for diagnostics only.
            theo_tflops = img_sec * 6.0 * total_params / 1e12
            theo_mfu = 100.0 * theo_tflops / args.peak_tflops

            print(
                f"[Epoch {epoch:03d}][Step {global_step:06d}] "
                f"loss={loss.item():.4f} lr={lr:.6f} mom={momentum:.5f} "
                f"img/sec={img_sec:.1f} "
                f"TFLOPS={tflops:.3f} MFU={mfu:.3f}% "
                f"theo_TFLOPS={theo_tflops:.3f} theo_MFU={theo_mfu:.3f}%",
                flush=True,
            )

            window_images = 0
            window_start = None

        global_step += 1

    if rank0 and window_images > 0 and window_start is not None:
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - window_start
        img_sec = window_images / max(elapsed, 1e-9)
        tflops = img_sec * args.flops_per_image / 1e12
        mfu = 100.0 * tflops / args.peak_tflops

        print(
            f"[Epoch {epoch:03d}][Final] "
            f"img/sec={img_sec:.1f} TFLOPS={tflops:.3f} MFU={mfu:.3f}%",
            flush=True,
        )

    return global_step


# ---------------------------------------------------------------------
# Args
# ---------------------------------------------------------------------
def get_args():
    p = argparse.ArgumentParser(description="Nano JEPA HPC trainer for Kaggle 2x T4")

    # Data
    p.add_argument("--data-path", type=str, default="./data")
    p.add_argument("--output-dir", type=str, default="./outputs")
    p.add_argument("--synthetic", action="store_true", help="Use synthetic data for MFU benchmarking")
    p.add_argument("--synthetic-size", type=int, default=100000)

    # Model
    p.add_argument("--img-size", type=int, default=64)
    p.add_argument("--patch-size", type=int, default=8)
    p.add_argument("--embed-dim", type=int, default=384)
    p.add_argument("--context-depth", type=int, default=6)
    p.add_argument("--target-depth", type=int, default=6)
    p.add_argument("--predictor-depth", type=int, default=3)
    p.add_argument("--heads", type=int, default=12)
    p.add_argument("--mlp-ratio", type=float, default=4.0)
    p.add_argument("--mask-ratio", type=float, default=0.75)

    # Training
    p.add_argument("--epochs", type=int, default=20)
    p.add_argument("--batch-size", type=int, default=256, help="Per-GPU batch size")
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--scale-lr", action="store_true", help="Linearly scale LR by global batch / 512")
    p.add_argument("--lr-scale-base-batch", type=int, default=512)
    p.add_argument("--weight-decay", type=float, default=0.05)
    p.add_argument("--warmup-epochs", type=int, default=2)
    p.add_argument("--momentum", type=float, default=0.996)
    p.add_argument("--momentum-end", type=float, default=1.0)

    # Loss regularization
    p.add_argument("--std-weight", type=float, default=0.05)
    p.add_argument("--cov-weight", type=float, default=0.04)
    p.add_argument("--disable-vicreg", action="store_true", help="Disable variance/covariance regularization")

    # Optimization
    p.add_argument("--clip-grad", type=float, default=0.0)
    p.add_argument("--workers", type=int, default=4)
    p.add_argument("--prefetch", type=int, default=4)
    p.add_argument("--log-interval", type=int, default=20)
    p.add_argument("--save-every", type=int, default=5)
    p.add_argument("--resume", type=str, default="")

    # Performance
    p.add_argument("--compile", action="store_true", help="torch.compile context encoder and predictor")
    p.add_argument("--compile-mode", type=str, default="max-autotune-no-cudagraphs")
    p.add_argument("--disable-grad-checkpointing", action="store_true")
    p.add_argument("--static-graph", action="store_true", default=True)
    p.add_argument("--no-static-graph", dest="static_graph", action="store_false")
    p.add_argument("--aug", action="store_true", default=True)
    p.add_argument("--no-aug", dest="aug", action="store_false")
    p.add_argument("--disable-fp16", action="store_true")

    # MFU accounting
    # Calibrated to match your reported Phase-1 MFU:
    #   985 img/sec * 23.1 MFLOPs/img ~= 22.75 GFLOPS => 0.035% of 65 TFLOPS.
    p.add_argument("--flops-per-image", type=float, default=23.1e6)
    p.add_argument("--peak-tflops", type=float, default=65.0)

    # Benchmark / eval
    p.add_argument("--benchmark-steps", type=int, default=0, help="Stop after N global steps")
    p.add_argument("--tsne", action="store_true")
    p.add_argument("--tsne-samples", type=int, default=500)

    p.add_argument("--seed", type=int, default=42)

    return p.parse_args()


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main():
    global torch
    args = get_args()

    if args.disable_vicreg:
        args.std_weight = 0.0
        args.cov_weight = 0.0

    rank, world_size, local_rank, ddp = setup_dist()
    device = torch.device(f"cuda:{local_rank}")

    set_seed(args.seed, rank)

    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    if hasattr(torch.backends.cuda, "enable_flash_sdp"):
        torch.backends.cuda.enable_flash_sdp(True)
        torch.backends.cuda.enable_mem_efficient_sdp(True)
        torch.backends.cuda.enable_math_sdp(True)

    if rank == 0:
        Path(args.output_dir).mkdir(parents=True, exist_ok=True)

    global_batch = args.batch_size * world_size
    base_lr = args.lr
    if args.scale_lr:
        base_lr = args.lr * float(global_batch) / float(args.lr_scale_base_batch)

    if rank == 0:
        print("=" * 80)
        print("Nano JEPA HPC Trainer")
        print("=" * 80)
        print(f"DDP enabled          : {ddp}")
        print(f"World size           : {world_size}")
        print(f"Per-GPU batch size   : {args.batch_size}")
        print(f"Global batch size    : {global_batch}")
        print(f"Base LR              : {base_lr}")
        print(f"FP16                 : {not args.disable_fp16}")
        print(f"Gradient checkpoint  : {not args.disable_grad_checkpointing}")
        print(f"torch.compile        : {args.compile}")
        print(f"MFU FLOPs/image      : {args.flops_per_image:.3e}")
        print(f"Peak TFLOPS          : {args.peak_tflops}")
        print("=" * 80, flush=True)

    model = NanoJEPA(
        img_size=args.img_size,
        patch_size=args.patch_size,
        embed_dim=args.embed_dim,
        context_depth=args.context_depth,
        target_depth=args.target_depth,
        predictor_depth=args.predictor_depth,
        heads=args.heads,
        mlp_ratio=args.mlp_ratio,
        mask_ratio=args.mask_ratio,
        grad_ckpt=not args.disable_grad_checkpointing,
    ).to(device)

    # Better Conv2d memory layout for T4.
    model.patch_embed.weight.data = model.patch_embed.weight.data.contiguous(memory_format=torch.channels_last)
    model.target_patch_embed.weight.data = model.target_patch_embed.weight.data.contiguous(memory_format=torch.channels_last)

    if args.compile:
        try:
            import torch._dynamo
            torch._dynamo.config.suppress_errors = True
        except Exception:
            pass

        # Compile only the heavy trainable transformers, not the EMA target branch.
        model.context_encoder = torch.compile(model.context_encoder, mode=args.compile_mode)
        model.predictor = torch.compile(model.predictor, mode=args.compile_mode)

    if ddp:
        model = DDP(
            model,
            device_ids=[local_rank],
            output_device=local_rank,
            gradient_as_bucket_view=True,
            static_graph=args.static_graph,
            find_unused_parameters=False,
        )

    model_without_ddp = unwrap_model(model)

    total_params = sum(p.numel() for p in model_without_ddp.parameters())
    trainable_params = sum(p.numel() for p in model_without_ddp.parameters() if p.requires_grad)

    if rank == 0:
        print(f"Total parameters     : {total_params / 1e6:.2f}M")
        print(f"Trainable parameters : {trainable_params / 1e6:.2f}M", flush=True)

    # Optimizer
    no_decay_keys = ("bias", "norm", "pos_embed", "mask_token")
    decay_params = []
    no_decay_params = []

    for n, p in model_without_ddp.named_parameters():
        if not p.requires_grad:
            continue
        if any(k in n for k in no_decay_keys):
            no_decay_params.append(p)
        else:
            decay_params.append(p)

    param_groups = [
        {"params": decay_params, "weight_decay": args.weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

    try:
        optimizer = torch.optim.AdamW(
            param_groups,
            lr=base_lr,
            betas=(0.9, 0.95),
            fused=True,
        )
    except TypeError:
        optimizer = torch.optim.AdamW(
            param_groups,
            lr=base_lr,
            betas=(0.9, 0.95),
        )

    scaler = torch.cuda.amp.GradScaler(enabled=not args.disable_fp16)

    dataset, sampler = build_dataset(args, rank, world_size, ddp)
    loader = build_loader(dataset, sampler, args)

    total_steps = len(loader) * args.epochs
    if args.benchmark_steps > 0:
        total_steps = args.benchmark_steps

    warmup_steps = len(loader) * args.warmup_epochs
    if args.benchmark_steps > 0:
        warmup_steps = min(warmup_steps, max(1, total_steps // 5))

    start_epoch = 0

    if args.resume and Path(args.resume).is_file():
        if rank == 0:
            print(f"Resuming from {args.resume}")
        ckpt = torch.load(args.resume, map_location=device)
        model_without_ddp.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scaler.load_state_dict(ckpt["scaler"])
        start_epoch = ckpt["epoch"] + 1

    global_step = start_epoch * len(loader)

    for epoch in range(start_epoch, args.epochs):
        global_step = train_one_epoch(
            model=model,
            model_without_ddp=model_without_ddp,
            loader=loader,
            optimizer=optimizer,
            scaler=scaler,
            epoch=epoch,
            args=args,
            device=device,
            world_size=world_size,
            rank=rank,
            global_step=global_step,
            total_steps=total_steps,
            warmup_steps=warmup_steps,
            base_lr=base_lr,
            total_params=total_params,
        )

        if args.benchmark_steps > 0 and global_step >= args.benchmark_steps:
            break

        if rank == 0 and args.benchmark_steps == 0 and args.save_every > 0:
            if (epoch + 1) % args.save_every == 0 or (epoch + 1) == args.epochs:
                ckpt_path = Path(args.output_dir) / f"nano_jepa_epoch_{epoch:03d}.pth"
                save_checkpoint(ckpt_path, model, optimizer, scaler, epoch, args)
                print(f"Saved checkpoint: {ckpt_path}", flush=True)

        if ddp:
            dist.barrier()

    if args.tsne and rank == 0 and not args.synthetic and args.benchmark_steps == 0:
        run_tsne(model_without_ddp, args, device)

    cleanup_dist()


if __name__ == "__main__":
    main()

Overwriting train_nano_jepa.py


In [ ]:
!python train_nano_jepa.py \
  --data-path "/kaggle/input/datasets/paultimothymooney/blood-cells/dataset2-master/dataset2-master/images/TRAIN" \
  --img-size 224 --patch-size 16 \
  --embed-dim 768 --context-depth 12 --target-depth 12 --predictor-depth 4 \
  --heads 12 --mlp-ratio 4.0 \
  --batch-size 128 \
  --workers 4 \
  --prefetch 4 \
  --compile \
  --disable-grad-checkpointing \
  --flops-per-image 48e9

Nano JEPA HPC Trainer
DDP enabled          : False
World size           : 1
Per-GPU batch size   : 128
Global batch size    : 128
Base LR              : 0.0003
FP16                 : True
Gradient checkpoint  : False
torch.compile        : True
MFU FLOPs/image      : 4.800e+10
Peak TFLOPS          : 65.0
Total parameters     : 200.54M
Trainable parameters : 114.74M
/kaggle/working/train_nano_jepa.py:875: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=not args.disable_fp16)
/kaggle/working/train_nano_jepa.py:602: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=not args.disable_fp16):
W0720 17:03:56.386000 1413 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/select